# KPI & Executive Analytics

## Objective

The objective of this notebook is to transform analytical findings into business-oriented Key Performance Indicators (KPIs) that enable executives to monitor customer churn, identify high-risk customer segments, evaluate financial exposure, and support data-driven strategic decision-making.

The KPIs developed in this notebook will serve as the primary data source for the Streamlit dashboard and executive reporting.

## What are Key Performance Indicators (KPIs)?

Key Performance Indicators (KPIs) are measurable business metrics used to evaluate organizational performance against strategic objectives.

Within customer retention analytics, KPIs enable executives to continuously monitor customer behavior, identify emerging churn risks, measure the effectiveness of retention strategies, and make informed business decisions.

Unlike exploratory analysis, KPIs summarize business performance into concise, actionable metrics suitable for executive dashboards.

## Notebook Structure

1. Overall Customer KPIs
2. Customer Retention KPIs
3. Geographic Risk Analysis
4. Customer Engagement KPIs
5. High-Value Customer KPIs
6. Executive KPI Summary
7. Strategic Recommendations
8. Executive Dashboard Preview
9. Conclusion

## 1. Overall Customer KPIs

### Business Question

What is the overall health of the customer portfolio?

### Methodology

This section summarizes the overall customer base by calculating key business metrics related to customer acquisition, retention, and churn.

These indicators provide executives with a high-level overview of organizational performance.

In [4]:
import pandas as pd 
df = pd.read_csv('../data/cleaned_bank_churn.csv')

In [5]:
total_customers = len(df)

churned_customers = df["Exited"].sum()

retained_customers = total_customers-churned_customers

overall_churn_rate = round(
    df["Exited"].mean()*100,
    2
)

retention_rate = round(
    100-overall_churn_rate,
    2
)

overall = pd.DataFrame({
    "KPI":[
        "Total Customers",
        "Churned Customers",
        "Retained Customers",
        "Overall Churn Rate",
        "Retention Rate"
    ],
    "Value":[
        total_customers,
        churned_customers,
        retained_customers,
        f"{overall_churn_rate}%",
        f"{retention_rate}%"
    ]
})

overall

,KPI,Value
0,Total Customers,10000
1,Churned Customers,2037
2,Retained Customers,7963
3,Overall Churn Rate,20.37%
4,Retention Rate,79.63%


## Findings

The bank currently serves 10,000 customers, of which approximately 20% have exited. The overall retention rate remains close to 80%, indicating that while the majority of customers remain with the bank, customer attrition continues to represent a significant business challenge.

## Business Implication

Overall churn rate serves as the primary indicator of customer retention performance. Continuous monitoring of this KPI enables management to evaluate the effectiveness of retention strategies and identify long-term trends in customer loyalty.

# 2. Customer Engagement KPIs

## Business Question

How does customer engagement influence customer retention, and are inactive customers more likely to churn?

## Methodology

Customer engagement is measured using the **IsActiveMember** indicator. Active and inactive customers are compared by calculating their respective population sizes, engagement rates, and churn rates.

This analysis evaluates whether customer activity is associated with customer retention.


In [6]:
active = df["IsActiveMember"].sum()

inactive = len(df)-active

active_rate = round(active/len(df)*100,2)

inactive_rate = round(inactive/len(df)*100,2)

active_churn = round(
    df[df["IsActiveMember"]==1]["Exited"].mean()*100,
    2
)

inactive_churn = round(
    df[df["IsActiveMember"]==0]["Exited"].mean()*100,
    2
)

## Findings

Inactive customers demonstrate a significantly higher churn rate than active customers, suggesting that customer engagement plays an important role in reducing customer attrition.

## Business Implication

Improving customer engagement should be considered a strategic priority. Personalized communication, digital banking adoption, loyalty programs, and proactive customer outreach may reduce churn among inactive customers.

# 3. Geographic Risk Index

## Business Question

Which geographic markets present the highest customer retention risk, and where should the bank prioritize retention efforts?

## Methodology

The churn rate is calculated for each country and compared with the overall churn rate to compute a Geographic Risk Index.

Risk Index = Regional Churn Rate ÷ Overall Churn Rate

A value greater than 1 indicates above-average churn risk, while values below 1 represent lower-than-average customer attrition.


In [7]:
geo = (
    df.groupby("Geography")
    .agg(
        Customers=("Exited","count"),
        Churn_Rate=("Exited",lambda x:x.mean()*100)
    )
)

geo["Risk_Index"] = (
    geo["Churn_Rate"]/overall_churn_rate
).round(2)

geo

,Customers,Churn_Rate,Risk_Index
Geography,,,
France,5014,16.154767,0.79
Germany,2509,32.443204,1.59
Spain,2477,16.673395,0.82


### Risk interpretation:

1. Risk Index > 1 → Above-average churn risk.
2. Risk Index = 1 → Average risk.
3. Risk Index < 1 → Below-average churn risk.

## Findings

Germany records the highest Geographic Risk Index, indicating that customers within this region are substantially more likely to churn compared to the overall customer population.

## Business Implication

Geographic regions exhibiting above-average churn risk should receive priority in customer retention planning. Regional marketing campaigns and localized customer engagement initiatives can help reduce customer attrition.

# 4. High-Value Customer KPIs

## Business Question

What level of financial risk is associated with customer churn among premium customers?

## Methodology

Premium customers are identified using account balance thresholds. The analysis calculates the number of premium customers, premium customer churn rate, average premium balance, and total balance associated with churned premium customers.

These indicators estimate the bank's financial exposure resulting from high-value customer attrition.


In [8]:
premium = df[df["Balance"]>=100000]

premium_customers = len(premium)

premium_churn = round(
    premium["Exited"].mean()*100,
    2
)

premium_percentage = round(
    premium_customers/len(df)*100,
    2
)

premium_loss = premium.loc[
    premium["Exited"]==1,
    "Balance"
].sum()

avg_premium_balance = round(
    premium["Balance"].mean(),
    2
)

## Findings

Although premium customers represent a relatively small proportion of the customer base, they account for a disproportionately large share of potential financial losses due to their higher account balances.

## Business Implication

Protecting premium customers should remain a strategic objective. Relationship managers, personalized financial services, and proactive retention programs can significantly reduce revenue loss associated with premium customer churn.

# 5. Financial Portfolio


## Business Question

What are the overall financial characteristics of the bank's customer portfolio?

## Methodology

This section summarizes customer financial information by calculating total deposits, average account balance, median balance, average estimated salary, and the proportion of customers maintaining zero account balances.

These KPIs provide an overview of the financial composition of the customer base.


In [9]:
financial = pd.DataFrame({
"KPI":[
"Total Deposits",
"Average Balance",
"Median Balance",
"Zero Balance Customers",
"Zero Balance Percentage"
],

"Value":[

df["Balance"].sum(),

round(df["Balance"].mean(),2),

round(df["Balance"].median(),2),

(df["Balance"]==0).sum(),

round((df["Balance"]==0).mean()*100,2)

]
})

financial

,KPI,Value
0,Total Deposits,7.648589e+08
1,Average Balance,7.648589e+04
2,Median Balance,9.719854e+04
3,Zero Balance Customers,3.617000e+03
4,Zero Balance Percentage,3.617000e+01


## Findings


A considerable proportion of customers maintain zero account balances, while the average account balance remains substantially higher than the median, indicating a positively skewed balance distribution.

## Business Implication

Understanding the financial profile of customers enables the bank to identify valuable customer segments, optimize product offerings, and develop targeted financial strategies for different customer groups.

# 6. Executive KPI Table

## Business Question

What are the most important business indicators that executives should monitor to evaluate customer retention performance?

## Methodology

The most critical KPIs generated throughout this notebook are consolidated into a single executive summary table. This dashboard-ready summary provides a concise overview of customer retention performance, financial exposure, customer engagement, and geographic risk.


In [10]:
executive = pd.DataFrame({

"KPI":[

"Overall Churn Rate",

"Retention Rate",

"Premium Churn",

"Revenue at Risk",

"Highest Risk Region"

],

"Value":[

f"{overall_churn_rate}%",

f"{retention_rate}%",

f"{premium_churn}%",

round(premium_loss,2),

geo["Risk_Index"].idxmax()

]

})

executive

,KPI,Value
0,Overall Churn Rate,20.37%
1,Retention Rate,79.63%
2,Premium Churn,25.23%
3,Revenue at Risk,159489691.0
4,Highest Risk Region,Germany


## Findings

The executive KPI summary highlights customer churn rate, retention rate, premium customer exposure, customer engagement levels, and geographic risk as the primary indicators requiring continuous monitoring.

## Business Implication

A centralized KPI dashboard enables management to monitor organizational performance efficiently, identify emerging retention risks, and support data-driven strategic decision-making.

---



In [11]:
overall.to_csv("../data/overall_kpis.csv",index=False)

financial.to_csv("../data/financial_kpis.csv",index=False)

geo.to_csv("../data/geographic_risk.csv")

executive.to_csv("../data/executive_kpis.csv",index=False)

---

# 7. Strategic Recommendations

## Business Question

Based on the analytical findings, what actions should the bank prioritize to improve customer retention and reduce financial risk?

## Methodology

Recommendations are formulated by integrating insights obtained from customer segmentation, churn analysis, financial assessment, and executive KPIs.

The objective is to translate analytical findings into practical business strategies that support long-term customer retention and organizational growth.

## Findings

The analysis identifies several customer segments requiring immediate attention, particularly inactive customers, premium customers, and customers residing in high-risk geographic regions.

## Business Implication

Implementing targeted retention campaigns, improving customer engagement, strengthening onboarding programs, expanding personalized financial services, and continuously monitoring executive KPIs will enable the bank to reduce customer churn and improve long-term profitability.

# 8. Executive Dashboard Preview

## Purpose

The analytical insights and Key Performance Indicators (KPIs) developed throughout this project are intended to support an interactive business intelligence dashboard built using Streamlit.

The dashboard will transform the analytical results into an intuitive decision-support system that enables executives, analysts, and business stakeholders to monitor customer retention performance in real time.

Rather than reviewing multiple notebooks, users will be able to explore customer churn through an interactive interface with dynamic filtering and drill-down capabilities.

---

## Dashboard Modules

The Streamlit dashboard will include the following core analytical modules:

### 1. Executive Overview

- Total Customers
- Churned Customers
- Retention Rate
- Overall Churn Rate
- Premium Customer Churn
- Revenue at Risk

These KPI cards provide an immediate overview of the bank's customer retention performance.

---

### 2. Customer Segmentation Dashboard

Interactive visualizations will allow users to analyze customer churn across multiple segments, including:

- Geography
- Gender
- Age Group
- Credit Score Band
- Balance Segment
- Tenure Group

This module enables rapid identification of high-risk customer populations.

---

### 3. Geographic Risk Dashboard

Executives will be able to compare customer churn across European regions using interactive visualizations and Geographic Risk Index metrics.

This module supports region-specific strategic planning and resource allocation.

---

### 4. Customer Engagement Dashboard

This section focuses on customer activity by comparing active and inactive customers.

Users will be able to evaluate:

- Customer engagement rate
- Active customer churn
- Inactive customer churn
- Engagement Drop Indicator

---

### 5. High-Value Customer Dashboard

This module highlights premium customer analytics, including:

- Premium customer population
- Premium churn rate
- Revenue at risk
- Average premium balance
- Premium customer financial exposure

This enables management to prioritize retention strategies for the bank's most valuable customers.

---

### 6. Interactive Filters

The dashboard will support dynamic filtering across multiple customer attributes, including:

- Geography
- Gender
- Age Group
- Credit Score Band
- Balance Segment
- Tenure Group
- Customer Activity Status

All visualizations and KPIs will update automatically based on the selected filters.

---

### 7. Drill-Down Analytics

Users will be able to perform multi-level exploration of customer segments.

Example workflow:

Germany
→ Age Group (46–60)
→ High Balance Customers
→ Churn Rate
→ Customer Profile Summary

This functionality enables detailed investigation of specific high-risk customer segments.

---

## Business Value

The interactive dashboard converts complex analytical findings into an accessible business intelligence platform.

By combining customer segmentation, executive KPIs, and dynamic visualizations, decision-makers can continuously monitor customer retention, identify emerging churn risks, evaluate financial exposure, and support evidence-based strategic planning.

The dashboard represents the final stage of the customer churn analytics workflow, transforming raw customer data into actionable business insights.

---

# 9. Conclusion

The KPI framework developed in this notebook transforms detailed customer analytics into concise business metrics suitable for executive monitoring and strategic decision-making.

These indicators summarize customer retention performance, customer engagement, financial exposure, and regional churn risk, providing the analytical foundation for the interactive Streamlit dashboard and ongoing business performance monitoring.